[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Balint-H/ssnr_sim/blob/main/SSNR2026/0_lif_neuron_exercises.ipynb)

# Tutorial 1: The Leaky Integrate-and-Fire (LIF) Neuron

**Workshop Day 1 — From single neurons to motor control signals**

---

## Why this tutorial?

The goal of this workshop is to simulate a **pool of motoneurons** that
converts a descending drive into **muscle excitation signals**. To get
there we need a neuron model, the **Leaky Integrate-and-Fire (LIF)** model {doc}`../docs/neuron`.

In this notebook we will build one from scratch, scale it up to a small pool, and see how
discrete spikes become smooth muscle excitation.

## What we will build 

| Section | What you do | Key concept |
|---|---|---|
| 1 | Define the LIF parameters | Membrane equation (ODE) |
| 2 | Generate an input current | Synaptic drive $I(t)$ |
| 3 | Integrate the membrane equation | Forward Euler method |
| 4 | Add spike detection and reset | Threshold mechanism |
| 5 | Add a refractory period | Biological realism |
| 6 | Wrap everything in `build_pool()` | From one neuron to many |
| 7 | Map spikes to muscle excitation | From discrete spikes to smooth drive |

## Prerequisites

- Python basics (loops, functions, NumPy arrays)
- A vague memory of what a differential equation is (we'll refresh it)

---
> **Solution:** [01_lif_neuron_solutions.ipynb](https://github.com/Balint-H/ssnr_sim/blob/main/SSNR2026/solutions/01_lif_neuron_solutions.ipynb)

---
## Setup

Run the cell below to import the required packages.


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print(f"NumPy {np.__version__} loaded.")


---
## Section 1: The LIF model

### The membrane equation

A biological neuron's membrane acts like an RC circuit: a capacitance
$C$ that stores charge, and a leak conductance $g_L$ that lets it
dissipate. The voltage across the membrane evolves as:

$$
C\,\frac{dV}{dt} = I(t) - g_L\,(V - E_L)
$$

Dividing both sides by $g_L$ and defining $\tau_m = C / g_L$ and
$R = 1 / g_L$, we get the standard form used throughout this notebook:

$$
\tau_m\,\frac{dV}{dt} = E_L - V + R\,I(t)
$$

### The reset rule

The LIF neuron fires a spike whenever $V$ reaches a threshold $V_{th}$,
and is immediately reset:

$$
V(t) \geq V_{th} \quad\Rightarrow\quad V(t) \leftarrow V_{reset}
$$

### Parameters

| Symbol | Meaning | Value |
|---|---|---|
| $\tau_m$ | membrane time constant | 20 ms |
| $E_L$ | leak (resting) potential | −60 mV |
| $V_{th}$ | spike threshold | −50 mV |
| $V_{reset}$ | reset potential | −70 mV |
| $R$ | membrane resistance ($1/g_L$) | 100 MΩ |
| $I(t)$ | input current | variable |


### Defining parameters

Define the simulation settings ($\Delta t$, duration) and the LIF
neuron parameters from the table above.



In [ ]:
# --- simulation ---
t_max = 150e-3   # total duration (s)
dt    = 1e-3     # timestep (s)

# --- LIF neuron ---
tau = 20e-3      # membrane time constant (s)
el  = -60e-3     # leak potential (V)
vr  = -70e-3     # reset potential (V)
vth = -50e-3     # spike threshold (V)
r   = 100e6      # membrane resistance (Ω)

# --- input current ---
i_0 = 25e-11     # mean current (A)
T   = 0.05       # sinusoid period (s)

print(t_max, dt, tau, el, vr, vth, r, i_0)


---
## Section 2: Simulating an input current

The neuron needs an input current $I(t)$ to do anything interesting.
We will define two versions and compare them throughout the notebook:

1. **Constant** — a flat current $I(t) = I_0$, useful as a baseline
2. **Sinusoidal** — an oscillating drive that mimics a rhythmic
   descending command:

$$
I(t) = I_0\!\left(1+\sin\!\left(\frac{2\pi}{T}\,t\right)\right)
$$

Note that this is always $\geq 0$ (it oscillates between $0$ and
$2I_0$), so we never inject negative current.

### Generating the two inputs
Define both input functions and plot them over the simulation time range.



In [ ]:
step_end = int(t_max / dt)
t_range = np.arange(step_end) * dt

def generate_sinusoidal_input(t_range, i_0, T):
    return i_0 * (1 + np.sin(2 * np.pi * t_range / T))

def generate_constant_input(t_range, i_0):
    return np.full(len(t_range), i_0)

i_trace       = generate_sinusoidal_input(t_range, i_0, T)
i_const_trace = generate_constant_input(t_range, i_0)


Plot both input traces to verify they look as expected before running the simulation.


In [ ]:
plt.figure()
plt.title('Input current $I(t)$')
plt.xlabel('time (s)')
plt.ylabel('$I$ (A)')
plt.plot(t_range, i_const_trace, linestyle='--', linewidth=0.9, label='constant ($I_0$)')
plt.plot(t_range, i_trace, label='sinusoidal')
plt.legend()
plt.show()


---
## Section 3: Discrete-time integration

We now simulate the evolution of the membrane equation in discrete time
steps. Starting from the ODE, we approximate the derivative with a
finite difference:

$$
\tau_m\,\frac{V(t+\Delta t)-V(t)}{\Delta t} = E_{L} - V(t) + R\,I(t)
$$

Rearranging to isolate $V(t + \Delta t)$:

$$
V(t + \Delta t) = V(t) + \frac{\Delta t}{\tau_m}\left( E_L - V(t) + R\, I(t) \right)
$$

This is the **forward Euler method**, the simplest way to numerically
integrate an ODE. We wrap it in a function that performs a **single
step**, so we can reuse it later.

### Coding Exercise 3: Simulating membrane potential

1. Complete `simulate_membrane` — one Euler step.

**Fill in the line marked `# TODO`.**


In [ ]:
def simulate_membrane(v, i, el=-60e-3, tau=20e-3, r=100e6, dt=1e-3):
    """Single forward-Euler step of the membrane equation.

    Parameters
    ----------
    v  : current membrane potential (V)
    i  : input current at this timestep (A)

    Returns
    -------
    v  : updated membrane potential (V)
    """
    # TODO: implement the Euler update from the equation above
    raise NotImplementedError


2. Run the simulation loop for both inputs, starting from $V(0) = E_L$.

**Fill in the lines marked `# TODO`.**


In [ ]:
v = el
v_trace = np.zeros(step_end)
for step, i in enumerate(i_trace):
    # TODO: call simulate_membrane and store the result
    raise NotImplementedError
    v_trace[step] = v

v_const = el
v_const_trace = np.zeros(step_end)
for step, i in enumerate(i_const_trace):
    # TODO: same as above for the constant input
    raise NotImplementedError
    v_const_trace[step] = v_const


3. Plot $V_m$ over time.


In [ ]:
plt.figure()
plt.title('$V_m$: sinusoidal vs constant input (no spiking yet)')
plt.xlabel('time (s)')
plt.ylabel('$V_m$ (V)')
plt.plot(t_range, v_trace, label='sinusoidal')
plt.plot(t_range, v_const_trace, linestyle='--', label='constant ($I_0$)')
plt.legend()
plt.show()


---
## Section 4: Adding spikes

Neurons fire an **action potential** when $V$ reaches $V_{th}$ and
the membrane is immediately reset to $V_{reset}$.

We add a `detect_spike` function that checks the threshold and returns
the (possibly reset) voltage together with a binary spike flag.

### Coding Exercise 4: Spike detection

1. Complete `detect_spike`.

**Fill in the lines marked `# TODO`.**


In [ ]:
def detect_spike(v, vth=-50e-3, vr=-70e-3):
    """Check if membrane potential crossed threshold.

    Returns
    -------
    v      : reset to vr if spike, unchanged otherwise
    spiked : 1 if spike occurred, 0 otherwise
    """
    # TODO: if v >= vth, return (vr, 1), otherwise return (v, 0)
    raise NotImplementedError


2. Re-run the simulation for both inputs, this time recording spike times.

**Fill in the lines marked `# TODO`.**


In [ ]:
# --- sinusoidal input ---
v = el
v_trace = np.zeros(step_end)
spike_times = []
spike_train = np.zeros(step_end)

for step, i in enumerate(i_trace):
    v = simulate_membrane(v, i)
    # TODO: call detect_spike, update v, and if spiked append step*dt to spike_times
    raise NotImplementedError
    v_trace[step] = v

print(f"Sinusoidal — spikes: {len(spike_times)}")


In [ ]:
# --- constant input ---
v_const = el
v_const_trace = np.zeros(step_end)
spike_times_const = []
spike_train_const = np.zeros(step_end)

for step, i in enumerate(i_const_trace):
    v_const = simulate_membrane(v_const, i)
    # TODO: same as above for the constant input
    raise NotImplementedError
    v_const_trace[step] = v_const

print(f"Constant — spikes: {len(spike_times_const)}")


3. Plot $V_m$ for both inputs with spiking active.


In [ ]:
plt.figure()
plt.title('LIF with spiking: sinusoidal vs constant input')
plt.xlabel('time (s)')
plt.ylabel('$V_m$ (V)')
plt.plot(t_range, v_trace, linewidth=1.2, label='sinusoidal')
plt.plot(t_range, v_const_trace, linewidth=1.2, linestyle='--', label='constant ($I_0$)')
for ts in spike_times:
    plt.axvline(ts, color='C0', alpha=0.2, linewidth=0.8)
for ts in spike_times_const:
    plt.axvline(ts, color='C1', alpha=0.2, linewidth=0.8)
plt.legend(loc='lower right', fontsize=8)
plt.show()


4. Plot a raster to compare the spike-time distributions of the two input conditions side by side.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 3), sharex=True)

axes[0].eventplot(spike_times, linewidths=1.2)
axes[0].set_ylabel('sinusoidal')
axes[0].set_yticks([])

axes[1].eventplot(spike_times_const, linewidths=1.2, colors='C1')
axes[1].set_ylabel('constant')
axes[1].set_yticks([])
axes[1].set_xlabel('time (s)')

fig.suptitle('Raster plot — single neuron, two inputs')
plt.tight_layout()
plt.show()


---
## Section 5: Refractory period

After firing, real neurons enter a brief **refractory period**
$t_{\mathrm{ref}}$ during which the membrane is clamped at $V_{reset}$
and no new spike can occur.

We combine `simulate_membrane` and `detect_spike` into a single
function `detect_spike_refractory` that handles the full timestep:
refractory check → Euler step → threshold check.

### Coding Exercise 5: Adding a refractory period

1. Complete `detect_spike_refractory` and compare the output with and
without the refractory period.

**Hint:** the neuron is refractory at time $t$ if `t − last_spike_time < t_ref`.
During the refractory period, return `(vr, 0, last_spike_time)` without
integrating.

**Fill in the lines marked `# TODO`.**


In [ ]:
def detect_spike_refractory(v, i, t, last_spike_time, t_ref=10e-3,
                             vth=-50e-3, vr=-70e-3, el=-60e-3,
                             tau=20e-3, r=100e6, dt=1e-3):
    """Single timestep: refractory check + Euler step + spike detection.

    Returns
    -------
    v               : updated membrane potential
    spiked          : 1 if spike occurred, 0 otherwise
    last_spike_time : updated if spike occurred, unchanged otherwise
    """
    # TODO step 1: t - last_spike_time < t_ref, return (vr, 0, last_spike_time)

    # TODO step 2: call simulate_membrane to get the new v

    # TODO step 3: call detect_spike to check for threshold crossing

    # TODO step 4: if spiked, update last_spike_time = t

    # TODO step 5: return (v, spiked, last_spike_time)
    raise NotImplementedError


2. Run the simulation with a 10 ms refractory period applied to the sinusoidal input.


In [ ]:
t_ref = 10e-3

v = el
v_trace_ref = np.zeros(step_end)
spike_times_ref = []
spike_train_ref = np.zeros(step_end)
last_spike_time = -np.inf

for step, i in enumerate(i_trace):
    t = step * dt
    v, spiked, last_spike_time = detect_spike_refractory(
        v, i, t, last_spike_time, t_ref)
    if spiked:
        spike_times_ref.append(t)
        spike_train_ref[step] = 1
    v_trace_ref[step] = v

print(f"With refractory — spikes: {len(spike_times_ref)} "
      f"| rate: {len(spike_times_ref)/t_max:.1f} Hz")


3. Plot $V_m$ with and without the refractory period to see how the clamping suppresses extra spikes.


In [ ]:
plt.figure()
plt.title('Effect of refractory period (sinusoidal input)')
plt.xlabel('time (s)')
plt.ylabel('$V_m$ (V)')
plt.plot(t_range, v_trace, linewidth=1.2, linestyle='--',
         label='without refractory')
plt.plot(t_range, v_trace_ref, linewidth=1.2,
         label='with refractory (10 ms)')
plt.legend(loc='lower right', fontsize=8)
plt.show()


### Firing rate

The **firing rate** is the number of spikes per unit time:

$$
r = \frac{\text{spike count}}{T} \quad \text{(Hz)}
$$

The refractory period caps it at $1 / t_{\mathrm{ref}}$ — with
$t_{\mathrm{ref}} = 10$ ms, no neuron can exceed 100 Hz.

4. Compute the firing rates for both conditions to quantify the effect of the refractory period.


In [ ]:
fr_no_ref = len(spike_times) / t_max
fr_ref = len(spike_times_ref) / t_max

print(f"Firing rate (no refractory):   {fr_no_ref:.1f} Hz")
print(f"Firing rate (with refractory): {fr_ref:.1f} Hz")


---
## Section 6: From one neuron to a pool

The neuromuscular system relies on
a **pool** of motoneurons firing together. Each neuron in the pool
receives the same common drive $I(t)$ plus independent noise,
which decorrelates their spike trains.

We wrap everything into `build_pool()` — it runs the single-neuron
loop from Section 5 once per neuron and returns a binary spike-train
matrix of shape `(N, timesteps)`.

### Coding Exercise 6: Simulating a pool

1. Read `build_pool` and make sure you understand how it reuses
   `detect_spike_refractory` inside a double loop (neurons × time).


In [ ]:
def build_pool(i_trace, N, noise_std, dt=1e-3, tau=20e-3,
               el=-60e-3, vr=-70e-3, vth=-50e-3,
               r=100e6, t_ref=10e-3):
    """Simulate a pool of N LIF neurons.

    Each neuron receives i_trace (common) + independent Gaussian noise.
    Input is clipped to >= 0 (no negative current).

    Returns
    -------
    spike_trains_pool : (N, step_end) binary array
    """
    step_end = len(i_trace)
    spike_trains_pool = np.zeros((N, step_end))

    for n in range(N):
        v = el
        last_spike_time = -np.inf
        for step, i_common in enumerate(i_trace):
            t = step * dt
            i = np.clip(i_common + np.random.randn() * noise_std, 0, None)
            v, spiked, last_spike_time = detect_spike_refractory(
                v, i, t, last_spike_time, t_ref)
            spike_trains_pool[n, step] = spiked

    return spike_trains_pool


2. Run it with `N = 50` neurons.


In [ ]:
N = 50
noise_std = 5e-11

spike_trains_pool = build_pool(i_trace, N, noise_std)

print(f"Pool of {N} neurons — mean firing rate: "
      f"{spike_trains_pool.sum() / (N * t_max):.1f} Hz")


3. The plot below shows what a single neuron actually receives: the common
sinusoidal drive plus a sample of Gaussian noise.


In [ ]:
i_noisy = np.clip(i_trace + np.random.randn(step_end) * noise_std, 0, None)

plt.figure()
plt.title('Input to a single neuron')
plt.xlabel('time (s)')
plt.ylabel('$I$ (A)')
plt.plot(t_range, i_trace, linewidth=1.2, label='common sinusoidal')
plt.plot(t_range, i_noisy, linewidth=0.7, alpha=0.7, label='sinusoidal + noise')
plt.legend()
plt.show()


4. Plot a raster for the full pool.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for n in range(N):
    times = t_range[spike_trains_pool[n] == 1]
    ax.scatter(times, np.full_like(times, n),
               marker='|', s=15, c='k', linewidths=0.8)
ax.set_xlabel('time (s)')
ax.set_ylabel('neuron #')
ax.set_title(f'Raster plot — pool of {N} LIF neurons')
plt.tight_layout()
plt.show()


---
## Section 7: From spikes to muscle excitation

In the neuromuscular system each motoneuron innervates a specific
muscle. A **mapping matrix** $\mathbf{M}$ (neurons × muscles) encodes
which neuron drives which muscle, and the excitation signal $E_m(t)$
for each muscle is obtained by smoothing the mapped spike counts with
a first-order low-pass filter:

$$
E_m(t + \Delta t) = \left(1 - \frac{\Delta t}{\tau_{exc}}\right) E_m(t)
\;+\; \frac{\Delta t}{\tau_{exc}} \sum_n M_{n,m}\, s_n(t)
$$

where $s_n(t) \in \{0,1\}$ is the spike output of neuron $n$ and
$\tau_{exc}$ controls how quickly excitation builds up and decays.

This is the bridge between discrete neural spikes and the smooth
signals that actually drive muscles.

### Coding Exercise 7: Computing muscle excitation

1. Complete `update_excitation` — a single-step filter, analogous to
   `simulate_membrane`.

**Fill in the lines marked `# TODO`.**


In [ ]:
def update_excitation(spikes, E_prev, mu_mapping, dt, tau_exc=20e-3):
    """One-step excitation update for all muscles.

    Parameters
    ----------
    spikes     : (n_neurons,) binary spike vector at current timestep
    E_prev     : (n_muscles,) excitation at previous timestep
    mu_mapping : (n_neurons, n_muscles) which neuron drives which muscle
    dt         : timestep (s)
    tau_exc    : excitation time constant (s)

    Returns
    -------
    E_new : (n_muscles,) updated excitation, clipped >= 0
    """
    
    musc_input = mu_mapping.T @ spikes
    # TODO step 1: compute alpha = dt / tau_exc
    E_new = 0
    # TODO step 2: E_new = (1 - ...) * E_prev + ... * musc_input
    return np.maximum(E_new, 0.0)



2. Define a mapping that splits the pool into 2 muscles.




In [ ]:
n_muscles = 2
mu_mapping = np.zeros((N, n_muscles))
mu_mapping[:N // 2, 0] = 1.0   # first half  -> muscle 0
mu_mapping[N // 2:, 1] = 1.0   # second half -> muscle 1

3. Run the excitation loop and plot the result against the input.


In [ ]:
# --- run excitation step-by-step ---
E = np.zeros(n_muscles)
E_trace = np.zeros((step_end, n_muscles))

for step in range(step_end):
    spikes = spike_trains_pool[:, step]
    E = update_excitation(spikes, E, mu_mapping, dt)
    E_trace[step] = E

# --- plot ---
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

axes[0].set_title('Input current')
axes[0].plot(t_range, i_trace, color='C1')
axes[0].set_ylabel('$I$ (A)')

axes[1].set_title('Muscle excitation from spike trains')
axes[1].plot(t_range, E_trace[:, 0], label='muscle 0', linewidth=1.5)
axes[1].plot(t_range, E_trace[:, 1], label='muscle 1', linewidth=1.5)
axes[1].set_ylabel('excitation (a.u.)')
axes[1].set_xlabel('time (s)')
axes[1].legend()

plt.tight_layout()
plt.show()
